In [1]:
# DO NOT CONTAINERISE
# =====
# | Domain | Phylum | Class | Order | Family | Genus | Species |
# | ------ | ------ | ----- | ----- | ------ | ----- | ------- |
# | 域     | 门     | 纲     | 目    | 科     | 属    | 种      |
# -----

# Dependency
# -----
# ! pip install -r requirements.txt
# ! pip list
# ! conda list

# !conda install -y requests
# !conda install -y nest_asyncio

# !conda install -y geopandas=0.10.2
# !conda install -y rdflib=6.1.1

# !pip show requests
# !pip show nest_asyncio

# !pip show geopandas
# !pip show rdflib

import os
import sys
import glob
import shutil
from pathlib import Path
from datetime import datetime

from io import StringIO
import zipfile
import asyncio
import requests
from urllib import parse
import json.decoder

import csv
import pandas as pd

import nest_asyncio

# base settings
# -----
conf_vlab_name     = "DNA"
conf_workflow_name = "PEMA"

# conf_workflow_id   = f"wid-{datetime.now().strftime('%Y%m%d_%H%M%S%f')}"

param_gene_sequences = "SRR3231901"
conf_workflow_id     = "wid-test_dataset_99-PEMA_Source_code_v.2.1.4"
case_id              = "cid-test_dataset_99-PEMA_Source_code_v.2.1.4"

# dev
# -----
# library: --volume="//c/DockerShare/DNA:/home/jovyan" naavre-fl-dna-jupyter:local
# NaaVRE: /home/jovyan/Virtual Labs/DNA/Git public
# conf_dir_code = os.path.join("/", "home", "jovyan", "Virtual Labs", conf_vlab_name, "Git public", "library")
# if not os.path.exists(conf_dir_code):
#     os.makedirs(conf_dir_code)

# conf_dir_data  = os.path.join("/", "home", "jovyan", "Cloud Storage", "naa-vre-user-data", conf_vlab_name, conf_workflow_id)
# if not os.path.exists(conf_dir_data):
#     os.makedirs(conf_dir_data)

# local
# -----
conf_dir_workspace = os.path.join("/", "home", "jovyan", "Cloud Storage")

conf_dir_data_local_tmp = os.path.join("/", "tmp", "data")
if not os.path.exists(conf_dir_data_local_tmp):
    os.makedirs(conf_dir_data_local_tmp)

# MINIO
# -----
conf_minio_public_bucket      = "naa-vre-public"
conf_minio_public_bucket_root = f"vl-{conf_vlab_name.lower()}"
conf_minio_public_local_root  = os.path.join(conf_dir_workspace, conf_minio_public_bucket, conf_minio_public_bucket_root)
conf_minio_public_local_code  = os.path.join(conf_dir_workspace, conf_minio_public_bucket, conf_minio_public_bucket_root, "code")
conf_minio_public_local_data  = os.path.join(conf_dir_workspace, conf_minio_public_bucket, conf_minio_public_bucket_root, "data", conf_workflow_name)
# if not os.path.exists(conf_minio_public_local_root):
#     os.makedirs(conf_minio_public_local_root)

conf_minio_user_bucket        = "naa-vre-user-data"
# conf_minio_user_bucket_root   = param_user_email
conf_minio_user_bucket_root   = conf_vlab_name
conf_minio_user_local_root    = os.path.join(conf_dir_workspace, conf_minio_user_bucket,   conf_minio_user_bucket_root)
conf_minio_user_local_code    = os.path.join(conf_dir_workspace, conf_minio_user_bucket,   conf_minio_user_bucket_root,   "library")
conf_minio_user_local_data    = os.path.join(conf_dir_workspace, conf_minio_user_bucket,   conf_minio_user_bucket_root,   f"{conf_workflow_name}-{conf_workflow_id}")
conf_minio_user_local_flog    = os.path.join(conf_minio_user_local_data, "log.md")
if not os.path.exists(conf_minio_user_local_root):
    os.makedirs(conf_minio_user_local_root)

if not os.path.exists(conf_minio_user_local_data):
    os.makedirs(conf_minio_user_local_data)
with open(conf_minio_user_local_flog, "w+") as fp_log:
    fp_log.write(f"# {conf_workflow_id}\n") 

# for workflow step
# .....
# if os.path.exists(conf_minio_user_local_flog):
#     with open(conf_minio_user_local_flog, "a+") as fp_log:
#         fp_log.write(f"\n## {workflow_step}\n") 
# else:
#     if not os.path.exists(conf_minio_user_local_data):
#         os.makedirs(conf_minio_user_local_data)
#     with open(conf_minio_user_local_flog, "w+") as fp_log:
#         fp_log.write(f"\n## {workflow_step}\n") 

# API key
# -----
# If running under NaaVRE, input `your api key` with the correct value and input in the GUI:
# secret_SERVICE_KEY = "d18e08911c964d45912eb1e954adf994"
# secret_SERVICE_KEY = SecretsProvider().set_secret("secret_SERVICE_KEY")
# secret_SERVICE_KEY = SecretsProvider().get_secret("secret_SERVICE_KEY")

# Input param
# -----
# workflow: 01, 02
# .....
# test dataset
# | id | dataset                                        | usage                                 | env | Local | NaaVRE | MyLifewatch      |
# | -- | ---------------------------------------------- | ------------------------------------- | --- | ----- | ------ | ---------------- |
# | 99 | SRR3231900,SRR3231901                          | PEMA,  Source code, v.2.1.4           | py  |       |        | fail: metamds    |
# | 00 | ERR7125480,ERR7125483,ERR7125486,ERR7125489    | WRiMS, Invasive_Checker               | py  |       |        | fail: PEMARunner |
# | 01 | ERR12541385,ERR12541386,ERR12541387,ERR7125481 | PEMA,  SequenceRetriever, mylifewatch | py  |       |        | fail: PEMARunner |
# | 02 | ERR2181465,ERR2181466,ERR2181467               | PEMA,  Runner, OTU                    | py  |       |        | fail: PEMARunner |
# | 03 | ERR3460470,ERR4018451,ERR4018452               | PEMA,  Converter                      | r   |       |        | fail: PEMARunner |

# # 99, test
# param_gene_sequences = "SRR3231900,SRR3231901"
# workflow_id          = "wid-test_dataset_99-PEMA_Source_code_v.2.1.4"
# case_id              = "cid-test_dataset_99-PEMA_Source_code_v.2.1.4"
# # 00
# param_gene_sequences = "ERR7125480,ERR7125483,ERR7125486,ERR7125489"
# workflow_id          = "wid-test_dataset_00-WRiMS_Invasive_Checker"
# case_id              = "cid-test_dataset_00-WRiMS_Invasive_Checker"
# 01
# param_gene_sequences = "ERR12541385,ERR12541386,ERR12541387,ERR7125481"
# workflow_id          = "wid-test_dataset_01-PEMA_SequenceRetriever"
# case_id              = "cid-test_dataset_01-PEMA_SequenceRetriever"
# # 02
# param_gene_sequences = "ERR2181465,ERR2181466,ERR2181467"
# workflow_id          = "wid-test_dataset_02-PEMA_Runner-OTU"
# case_id              = "cid-test_dataset_02-PEMA_Runner-OTU"
# # 03
# param_gene_sequences = "ERR3460470,ERR4018451,ERR4018452"
# workflow_id          = "wid-test_dataset_03-PEMA_Converter"
# case_id              = "cid-test_dataset_03-PEMA_Converter"

# PEMA-SequenceRetriever
# -----
# param_gene_sequences = "ERR3460470,ERR4018451,ERR4018452"  # user input, sep=","

conf_fname_seq_zip = "mydata.zip"
conf_fpath_seq_zip = "mydata"

# PEMA-Runner
# -----
# return: case_id
param_fname_par_tsv = "Template-parameters.tsv"
# add param_ for allowed settings in pema parameters.tsv
conf_fname_par_tsv = "parameters.tsv"

# OTU
# -----
# pema_otu_delimiter = "\t"
# bold_otu_delimiter = ","
conf_delimiter_tsv = "\t"
conf_delimiter_csv = ","

print("Finish: NaaVRE parameters")
print(f"Workspace public: {conf_minio_public_local_root}")
print(f"  {conf_minio_public_local_code}")
print(f"  {conf_minio_public_local_data}")

print(f"Workspace user: {conf_minio_user_local_root}")
print(f"  {conf_minio_user_local_code}")
print(f"  {conf_minio_user_local_data}")
print(f"  {conf_minio_user_local_flog}")


Finish: NaaVRE parameters
Workspace public: /home/jovyan/Cloud Storage/naa-vre-public/vl-dna
  /home/jovyan/Cloud Storage/naa-vre-public/vl-dna/code
  /home/jovyan/Cloud Storage/naa-vre-public/vl-dna/data/PEMA
Workspace user: /home/jovyan/Cloud Storage/naa-vre-user-data/DNA
  /home/jovyan/Cloud Storage/naa-vre-user-data/DNA/library
  /home/jovyan/Cloud Storage/naa-vre-user-data/DNA/PEMA-wid-test_dataset_99-PEMA_Source_code_v.2.1.4
  /home/jovyan/Cloud Storage/naa-vre-user-data/DNA/PEMA-wid-test_dataset_99-PEMA_Source_code_v.2.1.4/log.md


In [2]:
# PEMA-SequenceRetriever
# =====

# test in terminal
# python /home/jovyan/library/enaBrowserTools/python3_1.6/enaDataGet.py -f fastq -d /tmp/data/ena ERR12541385

# workflow: 01, 02, 03
# -----
import os
import sys
import glob

import zipfile

workflow_step = "PEMA-SequenceRetriever"

if os.path.exists(conf_minio_user_local_flog):
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 
else:
    if not os.path.exists(conf_minio_user_local_data):
        os.makedirs(conf_minio_user_local_data)
    with open(conf_minio_user_local_flog, "w+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 

# lib
# -----
dir_lib_ena = os.path.join(conf_minio_public_local_code,  "enaBrowserTools", "python3_1.6")
fname_lib_ena = "enaDataGet.py"
file_lib_ena = os.path.join(dir_lib_ena,  fname_lib_ena)

# input
# -----

# output
# -----
dir_pema = os.path.join(conf_minio_user_local_data, workflow_step)

dir_pema_ena = os.path.join(dir_pema)
if not os.path.exists(dir_pema_ena):
    os.makedirs(dir_pema_ena)

dir_pema_seq = os.path.join(conf_minio_user_local_data)
if not os.path.exists(dir_pema_seq):
    os.makedirs(dir_pema_seq)

fname_mydata_zip = f"{conf_fname_seq_zip}"
file_seq_zip     = os.path.join(dir_pema_seq, fname_mydata_zip)

# -----
genes_list = param_gene_sequences.split(',') if len(param_gene_sequences) else []

if (len(genes_list) == 0):
    raise RuntimeError("At least 1 gene sequence must be provided")

for gene in genes_list:
    # Run ENA browser tools
    # os.system(f"python {file_lib_ena} -f fastq -d {dir_pema_ena} {gene}")
    os.system(f'python "{file_lib_ena}" -f fastq -d "{dir_pema_ena}" -c {gene}')
    
    if len(os.listdir(os.path.join(dir_pema_ena, gene))) != 2:
        raise RuntimeError(f"Expected 2 files for gene `{gene}`. Found {len(os.listdir(dir_pema_ena))} @ {dir_pema_ena}")

file_fastq_list = glob.glob(os.path.join(dir_pema_ena, "*", "*.fastq.gz"))

with zipfile.ZipFile(file_seq_zip, "w") as zipObj:
    for file_fastq in file_fastq_list:
        zipObj.write(file_fastq, os.path.join(conf_fpath_seq_zip, os.path.basename(file_fastq)))
        print(f"Info: Compress gene file {file_fastq}")

print(f"Info: Generated compressed sequencing files {file_fastq}")

if not os.path.isfile(file_seq_zip):
    raise FileNotFoundError(str(file_seq_zip) + " is missing")

with open(conf_minio_user_local_flog, "a+") as fp_log:
    fp_log.write(f"\nFinish: {workflow_step}\n")
    fp_log.write(f"\nOutput: {file_seq_zip}\n")

print(f"Finish: {workflow_step}")


/home/jovyan/Cloud Storage/naa-vre-public/vl-dna/code/enaBrowserTools/python3_1.6/utils.py:103: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  sequence_pattern_1 = re.compile('^[A-Z]{1}[0-9]{5}(\.[0-9]+)?$')
/home/jovyan/Cloud Storage/naa-vre-public/vl-dna/code/enaBrowserTools/python3_1.6/utils.py:104: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  sequence_pattern_2 = re.compile('^[A-Z]{2}[0-9]{6}(\.[0-9]+)?$')
/home/jovyan/Cloud Storage/naa-vre-public/vl-dna/code/enaBrowserTools/python3_1.6/utils.py:105: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  wgs_sequence_pattern = re.compile('^[A-Z]{4}[0-9]{8,9}(\.[0-9]+)?$')
/home/jovyan/Cloud Storage/naa-vre-public/vl-dna/code/enaBrowserTo

Checking availability of https://www.ebi.ac.uk/ena/browser/api/xml/SRR3231901
SRR3231901_1.fastq.gz already exists in local directory, skipping
SRR3231901_2.fastq.gz already exists in local directory, skipping
Completed
Info: Compress gene file /home/jovyan/Cloud Storage/naa-vre-user-data/DNA/PEMA-wid-test_dataset_99-PEMA_Source_code_v.2.1.4/PEMA-SequenceRetriever/SRR3231901/SRR3231901_1.fastq.gz
Info: Compress gene file /home/jovyan/Cloud Storage/naa-vre-user-data/DNA/PEMA-wid-test_dataset_99-PEMA_Source_code_v.2.1.4/PEMA-SequenceRetriever/SRR3231901/SRR3231901_2.fastq.gz
Info: Generated compressed sequencing files /home/jovyan/Cloud Storage/naa-vre-user-data/DNA/PEMA-wid-test_dataset_99-PEMA_Source_code_v.2.1.4/PEMA-SequenceRetriever/SRR3231901/SRR3231901_2.fastq.gz
Finish: PEMA-SequenceRetriever


In [3]:
# PEMA-parameters
# =====

# workflow: 01, 02, 03
# -----
import os
import sys
import glob
from datetime import datetime

import zipfile

workflow_step_i = "PEMA-SequenceRetriever"
workflow_step   = "PEMA-parameters"

if os.path.exists(conf_minio_user_local_flog):
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 
else:
    if not os.path.exists(conf_minio_user_local_data):
        os.makedirs(conf_minio_user_local_data)
    with open(conf_minio_user_local_flog, "w+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 

# lib
# -----

# input
# -----
file_param_tsv_i = os.path.join(conf_minio_user_local_root, param_fname_par_tsv)

dir_mydata_zip   = os.path.join(conf_minio_user_local_data)
fname_mydata_zip = f"{conf_fname_seq_zip}"
file_mydata_zip  = os.path.join(dir_mydata_zip, fname_mydata_zip)

# output
# -----
file_param_tsv_o = os.path.join(conf_minio_user_local_data, conf_fname_par_tsv)

# set parameters
# -----
with open(file_param_tsv_i, 'r') as fp_i, open(file_param_tsv_o, 'w') as fp_o:
    for line_i in fp_i:
        line_o = line_i.strip()
        fp_o.write(f"{line_o}\n")

# zip "parameters.tsv" and "mydata"
# -----
# append "parameters.tsv" into "mydata.zip"

with zipfile.ZipFile(file_mydata_zip, "a") as zipObj:
    zipObj.write(file_param_tsv_o, os.path.basename(file_param_tsv_o))

with open(conf_minio_user_local_flog, "a+") as fp_log:
    fp_log.write(f"\nFinish: {workflow_step}\n")
    fp_log.write(f"\nOutput: {file_param_tsv_o}\n")
    fp_log.write(f"\nOutput: {file_mydata_zip}\n")

print(f"Finish: {workflow_step}")


Finish: PEMA-parameters


In [ ]:
# PEMA-Runner
# =====
# On UvA VM, https://pema-dev.naavre.net
# Codes are in README.md, tested on NaaVRE openLab

# workflow: 01, 02, 03
# -----
import os
import sys
import glob
from datetime import datetime

import zipfile
import requests

workflow_step_i = "PEMA-parameters"
workflow_step   = "PEMA-Runner"

if os.path.exists(conf_minio_user_local_flog):
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 
else:
    if not os.path.exists(conf_minio_user_local_data):
        os.makedirs(conf_minio_user_local_data)
    with open(conf_minio_user_local_flog, "w+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 

# lib
# -----

# input
# -----
dir_upload_zip   = os.path.join(conf_minio_user_local_data)
fname_mydata_zip = f"{conf_fname_seq_zip}"
file_upload_zip  = os.path.join(dir_upload_zip, fname_mydata_zip)
# print(dir_upload_zip)

# output
# -----
case_id   = ""
case_name = f"PEMA-{case_id}"
fname_result_zip  = ""
file_download_zip = ""

dir_pema = os.path.join(conf_minio_user_local_data, workflow_step)
if not os.path.exists(dir_pema):
    os.makedirs(dir_pema)
# print(dir_pema)

dir_unzip = os.path.join(conf_minio_user_local_data, workflow_step, "result")
if not os.path.exists(dir_unzip):
    os.makedirs(dir_unzip)
# print(dir_unzip)

# start
# -----
with open(file_upload_zip, 'rb') as fp_upload_zip, open(conf_minio_user_local_flog, "a+") as fp_log:
    str_print = f"Start: PEMA API, {case_name}, {datetime.now().strftime('%Y%m%d_%H%M%S%f')}"
    print(str_print)
    fp_log.write(f"{str_print}\n")

    # test api
    # .....
    # url = "https://pema-dev.naavre.net/pema/case/"
    # obj_response = requests.get(url)
    # dict_response = obj_response.json()
    # print(obj_response.status_code)
    # for key, val in dict_response.items():
    #     print("\n", key, "\n")
    #     for val_sub in val:
    #         print(val_sub, "\n")

    # Upload mydata and param
    # .....
    # output: case_id = f"cid-{}"
    url = "https://pema-dev.naavre.net/pema/case/"

    files = {'case_zip_file': (fname_mydata_zip, fp_upload_zip)}
    obj_response = requests.post(url, files=files)
    if obj_response.status_code == 200:
        dict_response = obj_response.json()
        case_id   = dict_response["case_id"]
        case_name = f"PEMA-{case_id}"
    
        str_print = f"Finish: Upload mydata and param, {case_name}, {datetime.now().strftime('%Y%m%d_%H%M%S%f')}"
        print(str_print)
        fp_log.write(f"{str_print}\n")
    else:
        print(obj_response.status_code)
        print(obj_response.text)

    # Run pema
    # .....
    # input: case_id
    url = f"https://pema-dev.naavre.net/pema/run/?case_id={case_id}"
    is_pema_run = False
    
    obj_response = requests.post(url, timeout=None)
    if obj_response.status_code == 200:
        # dict_response = obj_response.json()
        is_pema_run = True
        
        str_print = f"Finish: Run pema, {case_name}, {datetime.now().strftime('%Y%m%d_%H%M%S%f')}"
        print(str_print)
        fp_log.write(f"{str_print}\n")
    else:
        print(obj_response.status_code)
        print(obj_response.text)

    # Get pema result
    # .....
    # input: case_id
    url = f"https://pema-dev.naavre.net/pema/result/?case_id={case_id}"

    if is_pema_run:
        fname_result_zip  = f"{case_name}.zip"
        file_download_zip = os.path.join(dir_pema, fname_result_zip)
        
        # with zipfile.ZipFile(file_download_zip, 'r') as zipObj:
        #     zipObj.extractall(dir_unzip)
        
        obj_response = requests.get(url, stream=True)
        if obj_response.status_code == 200:
            # dict_response = obj_response.json()
            
            chunk_size = 128
            with open(file_download_zip, 'wb') as fd:
                for chunk in obj_response.iter_content(chunk_size=chunk_size):
                    fd.write(chunk)
        
            if os.path.isfile(file_download_zip):
                with zipfile.ZipFile(file_download_zip, 'r') as zip_ref:
                    zip_ref.extractall(dir_unzip)
                
                str_print = f"Finish: Get pema result, {case_name}, {datetime.now().strftime('%Y%m%d_%H%M%S%f')}"
                print(str_print)
                fp_log.write(f"{str_print}\n")
                
                str_print = f"Output: {dir_unzip}"
                print(str_print)
                fp_log.write(f"{str_print}\n")
            else:
                raise FileNotFoundError(f"{file_download_zip} is missing")
        else:
            print(obj_response.status_code)
            print(obj_response.text)

with open(conf_minio_user_local_flog, "a+") as fp_log:
    fp_log.write(f"\nFinish: {workflow_step}\n")

print(f"Finish: {workflow_step}")


Start: PEMA API, PEMA-, 20260407_123714614889
Finish: Upload mydata and param, PEMA-20260407_123714994975, 20260407_123715147231


In [ ]:
# OTU-Unify
# =====

# workflow: 01, 02, 03
# -----
import os
import sys
import glob
from datetime import datetime

import zipfile
import requests
from pathlib import Path
import pandas as pd

workflow_step_i = "PEMA-Runner"
workflow_step   = "OTU-Unify"

if os.path.exists(conf_minio_user_local_flog):
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 
else:
    if not os.path.exists(conf_minio_user_local_data):
        os.makedirs(conf_minio_user_local_data)
    with open(conf_minio_user_local_flog, "w+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 

# lib
# -----

# input
# -----
dir_pema = os.path.join(conf_minio_user_local_data, workflow_step_i, "result")
print(dir_pema)

# file_i_pema_otu = os.path.join(dir_pema, 'final_table.tsv')
file_i_pema_otu = os.path.join(dir_pema, 'finalTable.tsv')
file_i_bold_otu = os.path.join(dir_pema, 'all_sequences_grouped.fa_output.csv')

# output
# -----
dir_otu = os.path.join(conf_minio_user_local_data, workflow_step)
if not os.path.exists(dir_otu):
    os.makedirs(dir_otu)

# file_o_otu_csv  = Path(os.path.join(dir_otu,  'otu.tsv'))
# file_o_otu_csv.touch()
file_o_otu_csv  = os.path.join(dir_otu,  'otu.tsv')

# -----
is_pema_exists = Path(file_i_pema_otu).is_file()
is_bold_exists = Path(file_i_bold_otu).is_file()

if not is_pema_exists:
    raise RuntimeError('It is mandatory to provide the PEMA TSV file.')

# -----
df_pema = pd.read_csv(file_i_pema_otu, sep=conf_delimiter_tsv)
# print("DevOps: df_pema-00")
print(df_pema)

if "Classification" in df_pema.columns:
    df_pema = df_pema.rename(columns={"Classification": "classification"})

# rename first column to `OTU`. In the case of PEMA its called `Otuamplicon`
df_pema = df_pema.rename(columns={df_pema.columns[0]: "OTU"})

# otu_df["classification"] = otu_df["classification"].apply(lambda x: str(x).split(";")[-1])

missing_row_indexes = df_pema.loc[
    (df_pema['classification'].str.contains("No hits"))
    | (df_pema['classification'].str.contains("Unknown"))
    | (df_pema['classification'].str.contains("root"))
].index
# print("DevOps: missing_row_indexes")
# print(missing_row_indexes)

if df_pema["OTU"].str.startswith("ASV").all():
    df_pema["OTU"] = df_pema["OTU"].str.split(":", expand=True)[1]

df_pema.drop(missing_row_indexes, inplace=True)
# print("DevOps: df_pema-01")
print(df_pema.describe())

# -----
if not is_bold_exists:
    df_pema.to_csv(file_o_otu_csv, sep=conf_delimiter_tsv, index=False, mode="w")

else:
    df_bold = pd.read_csv(file_i_bold_otu, sep=conf_delimiter_csv)
    df_bold = df_bold.rename(columns={"OtuID": "OTU"})

    df_bold["classification"] = (
        df_bold["phylum"].astype(str)
        + ";"
        + df_bold["class"].astype(str)
        + ";"
        + df_bold["order"].astype(str)
        + ";"
        + df_bold["family"].astype(str)
        + ";"
        + df_bold["genus"].astype(str)
        + ";"
        + df_bold["species"].astype(str)
    )

    missing_row_indexes = df_bold.loc[
        (df_bold["classification"].str.contains("nohit"))
    ].index

    df_bold.drop(missing_row_indexes, inplace=True)

    df_pema.drop("classification", inplace=True, axis=1)
    df_pema["OTU"] = df_pema["OTU"].str.replace("Otu", "")

    df_bold = df_bold[["ID", "OTU", "classification"]]
    df_bold["OTU"] = df_bold["OTU"].str.split("_", expand=True)[0]

    df_bold = pd.merge(df_pema, df_bold, on="OTU")
    df_bold.drop("OTU", inplace=True, axis=1)
    a = df_bold["ID"]
    df_bold.drop(labels=["ID"], axis=1, inplace=True)
    df_bold.insert(0, "ID", a)

    df_bold = df_bold.rename(columns={"ID": "OTU"})
    df_bold.to_csv(file_o_otu_csv, sep=conf_delimiter_tsv, index=False, mode="w")

with open(conf_minio_user_local_flog, "a+") as fp_log:
    fp_log.write(f"\nFinish: {workflow_step}\n")
    fp_log.write(f"\nOutput: {dir_otu}\n")

print(f"Finish: {workflow_step}")


In [ ]:
# WoRMS Taxonomic Checker wrapper 
# =====

# workflow: 01, 02, 03
# -----
import os
import sys
import glob
from datetime import datetime

import zipfile
import asyncio
import requests
from urllib import parse
import json.decoder

from pathlib import Path
import pandas as pd

import nest_asyncio
nest_asyncio.apply()

workflow_step_i = "OTU-Unify"
workflow_step   = "WoRMS"

if os.path.exists(conf_minio_user_local_flog):
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 
else:
    if not os.path.exists(conf_minio_user_local_data):
        os.makedirs(conf_minio_user_local_data)
    with open(conf_minio_user_local_flog, "w+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 

# lib
# -----

# input
# -----
dir_otu = os.path.join(conf_minio_user_local_data, workflow_step_i)
fname_worms_i = 'otu.tsv'
file_worms_i  = os.path.join(dir_otu, fname_worms_i)

# output
# -----
dir_worms = os.path.join(conf_minio_user_local_data, workflow_step)
if not os.path.exists(dir_worms):
    os.makedirs(dir_worms)

fname_worms_o = 'worms.tsv'
file_worms_o  = os.path.join(dir_worms, fname_worms_o)

# start
# -----
aphia_ids = []

url_AphiaIDByName               = "http://www.marinespecies.org/rest/AphiaIDByName"
url_AphiaDistributionsByAphiaID = "http://www.marinespecies.org/rest/AphiaDistributionsByAphiaID"

async def aphiaID_by_name(classification, move_up_in_tree):
    name = classification.split(";")
    url_param = "marine_only=false"

    if move_up_in_tree:
        # reverse list
        name = name[::-1]
    else:
        # use only the last one in the list
        name = [name[-1]]

    for n in name:
        url = f"{url_AphiaIDByName}/{parse.quote(n)}?{url_param}"
        resp = requests.get(url)

        with open(conf_minio_user_local_flog, "a+") as fp_log:
            try:
                id = int(resp.text)
                str_print = f"aphia id `{id}` found for `{n}` .({url})"
                print(str_print)
                fp_log.write(f"* {str_print}\n")
                
                return {"classification": classification, "aphia_id": id}
            except ValueError:
                str_print = f"no aphia id found for `{n}`. ({url})"
                print(str_print)
                fp_log.write(f"* {str_print}\n")
                
                continue

    return {"classification": classification, "aphia_id": None}


async def aphia_distributions_by_aphia_id(aphia_id):
    url = f"{url_AphiaDistributionsByAphiaID}/{aphia_id}"
    resp = requests.get(url)

    try:
        r = resp.json()
        location_set = set()

        with open(conf_minio_user_local_flog, "a+") as fp_log:
            str_print = f"Distribution found for aphia id `{aphia_id}`. ({url})"
            print(str_print)
            fp_log.write(f"  * {str_print}\n")

        for item in r:
            if item["establishmentMeans"] == "Alien":
                location_id = item["locationID"]
                location_id = location_id.split("/")[-1]
                location_set.add(location_id)
        return {
            "aphia_id": [aphia_id] * len(location_set),
            "location_id": list(location_set),
        }

    except json.decoder.JSONDecodeError:
        with open(conf_minio_user_local_flog, "a+") as fp_log:
            str_print = f"No distribution found for aphia id `{aphia_id}`. ({url})"
            print(str_print)
            fp_log.write(f"  * {str_print}\n")

        return {"aphia_id": [aphia_id], "location_id": [None]}
    except Exception as err:
        with open(conf_minio_user_local_flog, "a+") as fp_log:
            str_print = f"Raised silent exception {err} while requesting {url}"
            print(str_print)
            fp_log.write(f"  * {str_print}\n")

        return {"aphia_id": [aphia_id], "location_id": [None]}

async def main():
    classification_move_up = True

    # file_worms_i = '/mnt/inputs/sample.csv'
    # file_worms_o = '/mnt/outputs/worms.csv'

    df_otu_table = pd.read_csv(file_worms_i, delimiter=conf_delimiter_tsv)

    rows, _ = df_otu_table.shape

    with open(conf_minio_user_local_flog, "a+") as fp_log:
        str_print = f"### Removing missing No hits/Uknown from:"
        print(str_print)
        fp_log.write(f"{str_print}\n")

        str_print = f"{df_otu_table}"
        print(str_print)
        fp_log.write(f"\n```\n{str_print}\n```\n\n")

        str_print = f"N-rows: {rows}"
        print(str_print)
        fp_log.write(f"{str_print}\n")

    tasks = []

    for classification in set(df_otu_table.classification.values):
        tasks.append(
            asyncio.ensure_future(
                aphiaID_by_name(classification, move_up_in_tree=classification_move_up)
            )
        )

    results = await asyncio.gather(*tasks)

    try:
        df_otu_table = df_otu_table.merge(
            pd.DataFrame(results, columns=["classification", "aphia_id"]).set_index(
                "classification"
            ),
            on="classification",
        )
        rows, _ = df_otu_table.shape

        with open(conf_minio_user_local_flog, "a+") as fp_log:
            str_print = f"{df_otu_table}"
            print(str_print)
            fp_log.write(f"\n```\n{str_print}\n```\n\n")

            str_print = f"N-rows: {rows}"
            print(str_print)
            fp_log.write(f"{str_print}\n")

        # df_otu_table = df_otu_table.dropna()
        # rows, _ = df_otu_table.shape
        # print("Removing rows with missing aphia ids")
        # print(f"{df_otu_table}")
        # print(f"N-rows: {rows}")

    except ValueError:
        raise

    # -----
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        # df_otu_table["aphia_id"] = df_otu_table["aphia_id"].astype("int64")

        str_print = f"Running `AphiaDistributionsByAphiaID` for `{len(df_otu_table.aphia_id.values)}` aphia ids retrieved."
        print(str_print)
        fp_log.write(f"### {str_print}\n")

    tasks.clear()

    for aphia_id in set(df_otu_table.aphia_id.values):
        tasks.append(
            asyncio.ensure_future(aphia_distributions_by_aphia_id(aphia_id))
        )

    results = await asyncio.gather(*tasks)

    distribution_table = pd.DataFrame(columns=["aphia_id", "location_id"])

    # -----
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        str_print = f"Get aphia ids & rename"
        print(str_print)
        fp_log.write(f"### {str_print}\n")

    for res in results:
        part = pd.DataFrame(
            {"aphia_id": res["aphia_id"], "location_id": res["location_id"]}
        )
        distribution_table = pd.concat([distribution_table, part])

    try:
        # print("Removing rows with missing location ids")
        df_otu_table = df_otu_table.merge(distribution_table, on="aphia_id")

        # df_otu_table = df_otu_table.dropna()
        # print(f"{df_otu_table}")
        # print(f"N-rows: {rows}")
    except ValueError:
        raise

    if "location_id" not in df_otu_table.columns:
        df_otu_table["location_id"] = ""
    
    # it may be the case that some rows have the same ids,
    # in that case rename them in order for rv lab to work
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        dupls = df_otu_table.loc[df_otu_table["OTU"].duplicated(), :].index

        str_print = f"Renaming {len(dupls)} rows ids"
        print(str_print)
        fp_log.write(f"* {str_print}\n")

    i = 0
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        for dupl in dupls:
            n_name = df_otu_table.loc[dupl, "OTU"] + "_" + str(i)
            
            str_print = f"Renaming `{df_otu_table.loc[dupl, 'OTU']}` to `{n_name}`"
            print(str_print)
            fp_log.write(f"* {str_print}\n")

            df_otu_table.loc[dupl, "OTU"] = n_name
            i = i + 1

    # save to file
    # -----
    with open(file_worms_o, 'w') as file:
        df_otu_table.to_csv(file, index=False, sep=conf_delimiter_tsv)

# Call main
# -----
# TODO, QPan, loop.close(): raise RuntimeError("Cannot close a running event loop")
# loop = asyncio.get_event_loop()
# try:
#     loop.run_until_complete(main())
# except Exception as err:
#     print("Error: ", err)
# finally:
#     loop.run_until_complete(loop.shutdown_asyncgens())
#     print(loop)
#     loop.close()

asyncio.run(main())

with open(conf_minio_user_local_flog, "a+") as fp_log:
    fp_log.write(f"\nFinish: {workflow_step}\n")
    fp_log.write(f"\nOutput: {dir_worms}\n")

print(f"Finish: {workflow_step}")


In [ ]:
# WRiMS, Invasive Checker
# =====

# workflow: 01, 02, 03
# -----
import os
import sys
import glob
import shutil
from pathlib import Path
from datetime import datetime

from io import StringIO
import zipfile
import asyncio
import requests
from urllib import parse
import json.decoder

import csv
import pandas as pd

workflow_step_i = "WoRMS"
workflow_step   = "WRiMS"

if os.path.exists(conf_minio_user_local_flog):
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 
else:
    if not os.path.exists(conf_minio_user_local_data):
        os.makedirs(conf_minio_user_local_data)
    with open(conf_minio_user_local_flog, "w+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 

# lib
# -----
dir_lib_InvasiveChecker   = os.path.join(conf_minio_public_local_code,  "invasive_checker")
fname_lib_InvasiveChecker = "main.py"
file_lib_InvasiveChecker  = os.path.join(dir_lib_InvasiveChecker,  fname_lib_InvasiveChecker)

# input
# -----
dir_worms = os.path.join(conf_minio_user_local_root, workflow_step_i)
if not os.path.exists(dir_worms):
    os.makedirs(dir_worms)

fname_i_worms = 'worms.tsv'
file_i_worms  = os.path.join(dir_worms, fname_i_worms)

# output
# -----
dir_wrims = os.path.join(conf_minio_user_local_root, workflow_step)
if not os.path.exists(dir_wrims):
    os.makedirs(dir_wrims)

# fname_wrims_o = 'wrims.csv'
# file_wrims_o  = os.path.join(dir_wrims, fname_wrims_o)

# Get PEMA parameters file from github
# -----
url_github_csv = "https://raw.githubusercontent.com/arms-mbon/data_workspace/main/reformatted_data/lifewatch/ARMS4Tesseract_PEMA_data.csv"

fname_github_tsv = "ARMS4Tesseract_PEMA_data.tsv"
file_github_tsv  = os.path.join(dir_wrims, fname_github_tsv)

obj_response = requests.get(url_github_csv)
if obj_response.status_code != 200:
    raise RuntimeError('CSV file not found in ' + url_github_csv)

fobj_url_github_csv = csv.reader(StringIO(obj_response.text), delimiter=conf_delimiter_csv)

with open(file_github_tsv, 'w') as fp_tsv:
    writer = csv.writer(fp_tsv, delimiter=conf_delimiter_tsv)
    
    for row in fobj_url_github_csv:
        writer.writerow(row)

# Run WRiMS Invasive Checker
# -----
with open(conf_minio_user_local_flog, "a+") as fp_log:
    str_exec = f'python "{file_lib_InvasiveChecker}" -i "{file_i_worms}" -m "{file_github_tsv}" -o "{dir_wrims}"'
    print(str_exec)
    fp_log.write(f"\n```\n{str_exec}\n```\n")

    os.system(str_exec)

# Copy files
# shutil.copyfile(f"{dir_wrims}/final_table_WoRMSandWRIMS.csv", "/mnt/outputs/final_table_WoRMSandWRiMS.csv")
# shutil.copyfile(f"{dir_wrims}/ARMS4Tesseract_PEMA_data.csv",  "/mnt/outputs/ARMS4Tesseract_PEMA_data.csv")
# shutil.copyfile(f"{dir_wrims}/OTU_withAphia.csv",             "/mnt/outputs/OTU_withAphia.csv")

with open(conf_minio_user_local_flog, "a+") as fp_log:
    fp_log.write(f"\nFinish: {workflow_step}\n")
    fp_log.write(f"\nOutput: {dir_wrims}\n")

print(f"Finish: {workflow_step}")


In [ ]:
# PEMA-Converter
# =====
# use R scripot from R kernal ipynb

# workflow: 01, 03
# -----
# print("Finish: PEMA-Converter")


In [ ]:
# metamds-observations
# =====
# use R scripot from R kernal ipynb

# workflow: 01
# -----
# print("Finish: metamds-observations")


In [ ]:
# Plot-Trend-Analysis
# =====

# workflow: 02
# -----


In [ ]:
# Taxa2Dis
# =====

# workflow: 03
# -----


In [ ]:
# Taxondive
# =====

# workflow: 03
# -----
